# 08 — ConvTran model with Reduced Features (C1/C2) & Early Stopping

This notebook loads the pre-pooled (`pooled`) and autoencoder (`autoencoder`) features. It features:
1. **Automatic Mixed Precision (AMP)** for memory safety.
2. **Early Stopping**: Monitors Validation Weighted F1 and stops if it hasn't improved for 10 epochs.
3. **Automatic GPU Cleanup**: Tries to release memory on interruption and at startup.

In [1]:
# Memory Cleanup (Important for Windows/PyTorch)
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("GPU Cache Cleared.")

GPU Cache Cleared.


In [2]:
# Path fix: use this repo's src and ConvTran
from pathlib import Path
import sys
import logging
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
import copy

PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
convtran_root_str = str(PROJECT_ROOT / 'ConvTran')

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)
if convtran_root_str not in sys.path:
    sys.path.insert(0, convtran_root_str)

# Remove cached src if necessary
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_splits_dir, get_experiment_dir, get_experiment_output_dir, ensure_dir, get_reduced_dir
from src.data import CLASS_NAMES, load_labels
from src.data.splits import load_splits
from src.training.evaluation import compute_metrics, compute_per_class_metrics, save_metrics, save_confusion_matrix
from Models.model import model_factory

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
NUM_CLASSES = len(CLASS_NAMES)


In [3]:
# Load target splits
print("Loading targets and splits...")
idx_train, idx_val, idx_test = load_splits()

try:
    y_full = load_labels()
except FileNotFoundError:
    print("labels.npy not found; loading from raw dataset... (this may take a second)")
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()

if y_full.ndim == 2:
    y_idx = np.argmax(y_full, axis=1)
else:
    y_idx = y_full

y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None


Loading targets and splits...


In [4]:
def train_convtran_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, condition_name):
    set_global_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    exp_dir = ensure_dir(get_experiment_dir(condition_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(condition_name, checkpoints=True))
    
    data_shape = (X_train.shape[0], 1, X_train.shape[1])
    config = {
        'Net_Type': ['C-T'],
        'emb_size': 16,
        'dim_ff': 256,
        'num_heads': 8,
        'Fix_pos_encode': 'tAPE', # time Absolute Position Encoding
        'Rel_pos_encode': 'eRPE', # efficient Relative Position Encoding
        'dropout': 0.1,
        'Data_shape': data_shape,
        'num_labels': NUM_CLASSES
    }
    
    model = model_factory(config).to(device)
    
    # Cleanup old experiments/variables from GPU context
    gc.collect()
    torch.cuda.empty_cache()
    
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    EPOCHS = 100
    BATCH_SIZE = 64
    ACCUMULATION_STEPS = 2
    PATIENCE = 10 
    
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler()
    n_train = len(X_train)
    
    ckpt_path = ckpt_dir / "convtran.pt"
    if ckpt_path.exists():
        print(f"\n  --> Found existing checkpoint at {ckpt_path}. Skipping training loop and evaluating directly.")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        print(f"\n  --> No checkpoint found. Starting Training procedure...")
        best_val_f1 = -1.0
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(model.state_dict())
        eval_batch_size = 128
        X_val_tensor = torch.from_numpy(X_val).float().unsqueeze(1) if X_val is not None else None
        
        try:
            for ep in range(EPOCHS):
                # --- Training Block ---
                model.train()
                perm = np.random.permutation(n_train)
                total_loss = 0.0
                n_b = 0
                opt.zero_grad()
                
                pbar = tqdm(range(0, n_train, BATCH_SIZE), desc=f"Epoch {ep+1}/{EPOCHS}")
                for i, start in enumerate(pbar):
                    idx = perm[start:start+BATCH_SIZE]
                    bx = torch.from_numpy(X_train[idx]).float().unsqueeze(1).to(device)
                    by = torch.from_numpy(y_train[idx]).long().to(device)
                    
                    with torch.cuda.amp.autocast():
                        logits = model(bx)
                        loss = F.cross_entropy(logits, by)
                        loss = loss / ACCUMULATION_STEPS
                    
                    scaler.scale(loss).backward()
                    
                    if (i + 1) % ACCUMULATION_STEPS == 0:
                        scaler.step(opt)
                        scaler.update()
                        opt.zero_grad()
                    
                    total_loss += loss.item() * ACCUMULATION_STEPS
                    n_b += 1
                    pbar.set_postfix(loss=total_loss/max(n_b, 1))
                
                # --- Validation Block ---
                if X_val_tensor is not None:
                    model.eval()
                    y_pred_val_list = []
                    with torch.no_grad():
                        for i in range(0, len(X_val_tensor), eval_batch_size):
                            batch_X = X_val_tensor[i:i + eval_batch_size].to(device)
                            with torch.cuda.amp.autocast():
                                logits_val_batch = model(batch_X)
                            y_pred_val_list.append(logits_val_batch.argmax(dim=1).cpu().numpy())
                    
                    y_pred_val = np.concatenate(y_pred_val_list)
                    metrics_val = compute_metrics(y_val, y_pred_val, task="multiclass", labels=list(range(NUM_CLASSES)))
                    val_f1 = metrics_val["weighted_f1"]
                    print(f"    [Validation] Weighted F1: {val_f1:.4f}")
                    
                    if val_f1 > best_val_f1:
                        best_val_f1 = val_f1
                        epochs_no_improve = 0
                        best_model_wts = copy.deepcopy(model.state_dict())
                        torch.save(model.state_dict(), ckpt_path)
                    else:
                        epochs_no_improve += 1
                        if epochs_no_improve >= PATIENCE:
                            print(f"\n  [Early Stopping] No improvement for {PATIENCE} epochs. Stopping...")
                            break
        except KeyboardInterrupt:
            print("\n  [Interrupted] Attempting Graceful GPU memory cleanup...")
        finally:
            # Always reload best or clean up after crash
            model.load_state_dict(best_model_wts)
            print(f"  Best weights reloaded to GPU context from memory.")
            torch.cuda.empty_cache()
    
    # Final Evaluation
    model.eval()
    all_metrics = {}
    labels_list = list(range(NUM_CLASSES))
    eval_batch_size = 128
    
    with torch.no_grad():
        y_pred_test_list = []
        X_test_tensor = torch.from_numpy(X_test).float().unsqueeze(1)
        print("\nEvaluating final best model configuration on Test Set...")
        for i in tqdm(range(0, len(X_test_tensor), eval_batch_size), desc="Testing"):
            batch_X = X_test_tensor[i:i + eval_batch_size].to(device)
            with torch.cuda.amp.autocast():
                logits_test_batch = model(batch_X)
            y_pred_test_list.append(logits_test_batch.argmax(dim=1).cpu().numpy())
        y_pred_test = np.concatenate(y_pred_test_list)
    
    metrics_test = compute_metrics(y_test, y_pred_test, task="multiclass", labels=labels_list)
    per_class_test = compute_per_class_metrics(y_test, y_pred_test, labels=labels_list, target_names=CLASS_NAMES)
    all_metrics["weighted_f1"] = metrics_test["weighted_f1"]
    all_metrics["per_class"] = per_class_test
    
    save_metrics(all_metrics, exp_dir / f"{condition_name}_metrics.json")
    print(f"  Test weighted F1: {metrics_test['weighted_f1']:.4f}")
    
    # Final Cell Cleanup
    torch.cuda.empty_cache()
    gc.collect()
    return all_metrics, exp_dir

def print_metrics_summary(metrics, model_name):
    print("\n" + "=" * 60)
    print(f"  {model_name} — Test metrics")
    print("=" * 60)
    print(f"  Weighted F1: {metrics.get('weighted_f1', 0):.4f}")
    per = metrics.get("per_class", [])
    if per:
        print(f"  Per-class: {'Class':<12} {'Precision':>8} {'Recall':>8} {'F1':>8} {'Support':>8}")
        print("  " + "-" * 52)
        for p in per:
            print(f"  {p['class']:<12} {p['precision']:>8.3f} {p['recall']:>8.3f} {p['f1']:>8.3f} {p['support']:>8d}")
    print("=" * 60)

In [5]:
# -----------------------------------------------------
# C1: Pooling + ConvTran
# -----------------------------------------------------
print("=== C1: Pooling + ConvTran ===")
red_dir = get_reduced_dir() / "pooled"
X_train_c1 = np.load(red_dir / "train.npy").astype(np.float32)
X_test_c1  = np.load(red_dir / "test.npy").astype(np.float32)
X_val_c1   = np.load(red_dir / "val.npy").astype(np.float32) if (red_dir / "val.npy").exists() else None

metrics_c1, _ = train_convtran_and_evaluate(X_train_c1, X_val_c1, X_test_c1, y_train, y_val, y_test, "C1_pooled_convtran")
print_metrics_summary(metrics_c1, "C1 (Pooling + ConvTran)")

=== C1: Pooling + ConvTran ===


2026-04-14 13:22:07,937 | INFO | Global seed set to 0
C:\Users\hmanasi1\AppData\Local\anaconda3\envs\dl_project\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



  --> No checkpoint found. Starting Training procedure...


Epoch 1/100:   0%|                                                                             | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\hmanasi1\AppData\Local\anaconda3\envs\dl_project\lib\site-packages\torch\nn\modules\conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1026.)
  return F.conv2d(
Epoch 1/100: 100%|████████████████████████████████████████████████████████| 565/565 [38:52<00:00,  4.13s/it, loss=1.09]
C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.au

    [Validation] Weighted F1: 0.5094


...

    [Validation] Weighted F1: 0.7931


Epoch 46/100:   0%|                                                                            | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 46/100:   0%|                                                      | 1/565 [00:09<1:26:44,  9.23s/it, loss=0.524]



  [Interrupted] Attempting Graceful GPU memory cleanup...
  Best weights reloaded to GPU context from memory.

Evaluating final best model configuration on Test Set...


Testing:   0%|                                                                                  | 0/53 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Testing: 100%|█████████████████████████████████████████████████████████████████████████| 53/53 [06:43<00:00,  7.61s/it]
2026-04-16 17:14:08,224 | INFO | Saved metrics to C:\Users\hmanasi1\Documents\deeplearning\Deep-Learning-Project\experiments\C1_pooled_convtran\C1_pooled_convtran_metrics.json


  Test weighted F1: 0.8006

  C1 (Pooling + ConvTran) — Test metrics
  Weighted F1: 0.8006
  Per-class: Class        Precision   Recall       F1  Support
  ----------------------------------------------------
  AF              0.823    0.824    0.823     2986
  SVT             0.250    0.012    0.024      162
  Sinus Brady     0.844    0.924    0.882     2425
  Sinus Rhythm    0.708    0.662    0.684     1200


In [6]:
# -----------------------------------------------------
# C2: Autoencoder + ConvTran
# -----------------------------------------------------
print("\n=== C2: Autoencoder + ConvTran ===")
red_dir = get_reduced_dir() / "autoencoder"
X_train_c2 = np.load(red_dir / "train.npy").astype(np.float32)
X_test_c2  = np.load(red_dir / "test.npy").astype(np.float32)
X_val_c2   = np.load(red_dir / "val.npy").astype(np.float32) if (red_dir / "val.npy").exists() else None

metrics_c2, _ = train_convtran_and_evaluate(X_train_c2, X_val_c2, X_test_c2, y_train, y_val, y_test, "C2_autoencoder_convtran")
print_metrics_summary(metrics_c2, "C2 (Autoencoder + ConvTran)")

2026-04-16 17:14:13,956 | INFO | Global seed set to 0



=== C2: Autoencoder + ConvTran ===


C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



  --> No checkpoint found. Starting Training procedure...


Epoch 1/100:   0%|                                                                             | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/100: 100%|█████████████████████████████████████████████████████████| 565/565 [00:11<00:00, 48.92it/s, loss=1.1]
C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

    [Validation] Weighted F1: 0.8672


Epoch 77/100:   0%|                                                                            | 0/565 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 77/100: 100%|███████████████████████████████████████████████████████| 565/565 [00:13<00:00, 41.90it/s, loss=0.36]
C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


    [Validation] Weighted F1: 0.8628

  [Early Stopping] No improvement for 10 epochs. Stopping...
  Best weights reloaded to GPU context from memory.

Evaluating final best model configuration on Test Set...


Testing:   0%|                                                                                  | 0/53 [00:00<?, ?it/s]C:\Users\hmanasi1\AppData\Local\Temp\ipykernel_24764\1684357133.py:126: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Testing: 100%|████████████████████████████████████████████████████████████████████████| 53/53 [00:00<00:00, 160.03it/s]
2026-04-16 17:30:01,603 | INFO | Saved metrics to C:\Users\hmanasi1\Documents\deeplearning\Deep-Learning-Project\experiments\C2_autoencoder_convtran\C2_autoencoder_convtran_metrics.json


  Test weighted F1: 0.8652

  C2 (Autoencoder + ConvTran) — Test metrics
  Weighted F1: 0.8652
  Per-class: Class        Precision   Recall       F1  Support
  ----------------------------------------------------
  AF              0.877    0.863    0.870     2986
  SVT             0.575    0.142    0.228      162
  Sinus Brady     0.916    0.974    0.944     2425
  Sinus Rhythm    0.775    0.786    0.780     1200
